In [2]:
!pip install torch numpy matplotlib

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [10]:
def generate_data(N=10000):
    bits = np.random.randint(0,2,N)
    symbols = 2*bits

    y = np.zeros(N)

    for k in range(N):
        y[k] = symbols[k]
        if k-1 >= 0:
            y[k] += 0.5*symbols[k-1]
        if k-2 >= 0:
            y[k] += 0.3*symbols[k-2]

    noise = 0.2*np.random.randn(N)
    y += noise

    return bits, y

In [12]:
def create_sequences(y, bits, seq_len=5):
    X, Y = [], []

    for i in range(seq_len, len(y)):
        X.append(y[i-seq_len:i+1])  
        Y.append(bits[i])

    return np.array(X), np.array(Y)

In [14]:
bits, y = generate_data()

X, Y = create_sequences(y, bits)

X = (X - np.mean(X)) / (np.std(X) + 1e-8)
X = X.reshape(X.shape[0], X.shape[1], 1)

X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)

In [16]:
class ISI_LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=32, batch_first=True)
        self.fc = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return self.sigmoid(out)

In [18]:
model = ISI_LSTM()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [22]:
epochs = 50

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X)
    loss = criterion(outputs.view(-1), Y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6917
Epoch 2, Loss: 0.6907
Epoch 3, Loss: 0.6897
Epoch 4, Loss: 0.6887
Epoch 5, Loss: 0.6877
Epoch 6, Loss: 0.6868
Epoch 7, Loss: 0.6858
Epoch 8, Loss: 0.6848
Epoch 9, Loss: 0.6839
Epoch 10, Loss: 0.6829
Epoch 11, Loss: 0.6820
Epoch 12, Loss: 0.6810
Epoch 13, Loss: 0.6800
Epoch 14, Loss: 0.6791
Epoch 15, Loss: 0.6781
Epoch 16, Loss: 0.6771
Epoch 17, Loss: 0.6762
Epoch 18, Loss: 0.6752
Epoch 19, Loss: 0.6742
Epoch 20, Loss: 0.6732
Epoch 21, Loss: 0.6722
Epoch 22, Loss: 0.6712
Epoch 23, Loss: 0.6702
Epoch 24, Loss: 0.6692
Epoch 25, Loss: 0.6682
Epoch 26, Loss: 0.6671
Epoch 27, Loss: 0.6661
Epoch 28, Loss: 0.6650
Epoch 29, Loss: 0.6639
Epoch 30, Loss: 0.6628
Epoch 31, Loss: 0.6617
Epoch 32, Loss: 0.6605
Epoch 33, Loss: 0.6593
Epoch 34, Loss: 0.6581
Epoch 35, Loss: 0.6569
Epoch 36, Loss: 0.6557
Epoch 37, Loss: 0.6544
Epoch 38, Loss: 0.6531
Epoch 39, Loss: 0.6517
Epoch 40, Loss: 0.6503
Epoch 41, Loss: 0.6489
Epoch 42, Loss: 0.6474
Epoch 43, Loss: 0.6459
Epoch 44, Loss: 0.64

In [24]:
with torch.no_grad():
    preds = model(X).squeeze().numpy()

pred_bits = (preds > 0.5).astype(int)

In [26]:
def compute_ber(pred, true):
    errors = np.sum(pred != true)
    return errors / len(true)

ber = compute_ber(pred_bits, Y.numpy())
print("BER:", ber)

BER: 0.21660830415207605


In [28]:
simple_pred = (y[5:] > 0).astype(int)

ber_simple = compute_ber(simple_pred, bits[5:])
print("BER without LSTM:", ber_simple)

BER without LSTM: 0.4367183591795898


In [2]:
labels = ['No Equalizer', 'LSTM']
values = [ber_simple, ber]

plt.bar(labels, values)
plt.ylabel("BER")
plt.title("Performance Comparison")
plt.show()

NameError: name 'ber_simple' is not defined